# EDA of results of SF Model v3

In [1]:
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sb
import numpy as np
import os
from itertools import product

In [2]:
df = pd.read_csv("./ts_series_result.csv")
# Cleaning
def clean_key(col):
    return col.str.replace(r'[\[\]",]', '', regex=True).str.strip()

split_keys = df['SERIES'].str.split('\n', expand=True)
df['key_1'] = clean_key(split_keys[1])
df['key_2'] = clean_key(split_keys[2])
df['key_3'] = clean_key(split_keys[3])

df['as_of_date'] = pd.to_datetime(df['TS']).dt.date
df['value'] = df['Y']
df = df.drop(columns=['SERIES', 'TS', 'Y'])

df.head()

,FORECAST,LOWER_BOUND,UPPER_BOUND,IS_ANOMALY,PERCENTILE,DISTANCE,key_1,key_2,key_3,as_of_date,value
0,7285.171227,6898.982089,7671.360365,False,0.717364,0.575028,key_1_b,key_2_a,key_3_c,2025-07-07,7352.658754
1,7228.740861,6842.551723,7614.929999,False,0.912753,1.357905,key_1_b,key_2_a,key_3_c,2025-07-08,7388.109942
2,6883.174113,6496.984975,7269.363251,False,0.662169,0.418391,key_1_b,key_2_a,key_3_c,2025-07-09,6932.278078
3,6669.620598,6283.431460,7055.809737,False,0.889913,1.226066,key_1_b,key_2_a,key_3_c,2025-07-10,6813.516465
4,6661.688486,6275.499347,7047.877624,False,0.319146,-0.470088,key_1_b,key_2_a,key_3_c,2025-07-11,6606.517137


In [3]:
allowed_key_1 = ['key_1_a', 'key_1_b', 'key_1_c']
allowed_key_2 = ['key_2_a', 'key_2_b', 'key_2_c']    
allowed_key_3 = ['key_3_a', 'key_3_b', 'key_3_c']    

dfs = []

for key1,key2,key3 in product(allowed_key_1, allowed_key_2, allowed_key_3):
    filtered_df = df[
        (df['key_1'] == key1) & 
        (df['key_2'] == key2) & 
        (df['key_3'] == key3)
    ]
    if not filtered_df.empty:
        dfs.append(filtered_df)
    
dfs    

[         FORECAST  LOWER_BOUND  UPPER_BOUND  IS_ANOMALY  PERCENTILE  DISTANCE  \
 5642  7592.697382  7108.428461  8076.966303       False    0.099684 -1.283357   
 5643  7532.529481  7048.260560  8016.798402       False    0.672654  0.447254   
 5644  7751.453671  7267.184750  8235.722592       False    0.262184 -0.636626   
 5645  8058.755634  7574.486713  8543.024555       False    0.581696  0.206235   
 5646  8161.753390  7677.484469  8646.022311       False    0.656101  0.401845   
 ...           ...          ...          ...         ...         ...       ...   
 6001  7777.627040  7293.358119  8261.895961       False    0.427403 -0.182989   
 6002  8058.179620  7573.910699  8542.448541       False    0.441412 -0.147391   
 6003  8173.302166  7689.033245  8657.571088       False    0.156194 -1.010225   
 6004  8130.738698  7646.469777  8615.007619       False    0.691312  0.499573   
 6005  7613.691606  7129.422685  8097.960527       False    0.942794  1.578673   
 
         key_1

In [9]:
import plotly.graph_objects as go
import pandas as pd
import os

os.makedirs('./anomalies_plotly', exist_ok=True)

for idx, d in enumerate(dfs):
    if d.empty:
        continue
    
    d = d.sort_values('as_of_date')
    d['dist_upper'] = (d['value'] - d['UPPER_BOUND']).clip(lower=0)
    d['dist_lower'] = (d['value'] - d['LOWER_BOUND']).clip(upper=0)
    d['total_dist'] = d['dist_upper'] + d['dist_lower']

    k1, k2, k3 = d['key_1'].iloc[0], d['key_2'].iloc[0], d['key_3'].iloc[0]

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=pd.concat([d['as_of_date'], d['as_of_date'][::-1]]),
        y=pd.concat([d['UPPER_BOUND'], d['LOWER_BOUND'][::-1]]),
        fill='toself',
        fillcolor='rgba(128, 128, 128, 0.15)',
        line=dict(color='rgba(255,255,255,0)'),
        hoverinfo="skip",
        name='Safe Zone (PI)',
    ))

    fig.add_trace(go.Scatter(
        x=d['as_of_date'], 
        y=d['FORECAST'],
        name='Baseline Forecast',
        line=dict(color='rgba(255, 127, 14, 0.7)', width=1.5) # Solid, slightly transparent orange
    ))

    fig.add_trace(go.Scatter(
        x=d['as_of_date'], 
        y=d['value'],
        name='Actual Value',
        mode='lines',
        line=dict(color='#1f77b4', width=2),
        customdata=d[['UPPER_BOUND', 'LOWER_BOUND', 'total_dist']],
        hovertemplate=(
            "<b>Actual: %{y:,.2f}</b><br>" +
            "Upper: %{customdata[0]:,.2f}<br>" +
            "Lower: %{customdata[1]:,.2f}<br>" +
            "Dist from Bound: %{customdata[2]:,.2f}<extra></extra>"
        )
    ))

    anomalies = d[d['IS_ANOMALY'] == True]
    if not anomalies.empty:
        fig.add_trace(go.Scatter(
            x=anomalies['as_of_date'],
            y=anomalies['value'],
            name='Anomaly Detected',
            mode='markers',
            marker=dict(color='#d62728', size=5)
        ))

    fig.update_layout(
        title=f"<b>Dynamic Anomaly Detection</b><br><sup>{k1} | {k2} | {k3}</sup>",
        xaxis_title="Date",
        yaxis_title="Metric Value",
        template="plotly_white",
        width=1400,
        height=600,
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        xaxis=dict(showgrid=True, gridcolor='lightgray'),
        yaxis=dict(showgrid=True, gridcolor='lightgray')
    )

    fig.write_html(f'./anomalies_plotly/anomaly_plot_{idx}.html')
